# Lab bench — prototyping an extraction prompt

**This notebook is a workbench, not a product.** Nothing imports it. It is safe to
change wildly, leave half-finished, or delete. Its only job is to let you *see*
what Claude extracts from a document so you can tune the prompt by eye before
that tuning ever touches `extraction_engine.py` or a schema seed.

The production path (`app/services/extraction_engine.py`) runs unattended: OCR a
PDF → batch the schema fields → call Claude → normalise → write rows. You never
watch it. Here you watch every step, react to it, and iterate. That is the whole
difference between a notebook and a `.py` file.

**How to run it**
1. `cd backend && pip install jupyter` (Jupyter is a dev tool, not a backend dep)
2. `jupyter lab` (or open this file in VS Code) and run the cells top to bottom.
3. It reads your real `backend/.env`, so a valid `ANTHROPIC_API_KEY` must be set.


## Step 1 — wire up to the *real* client

We reuse the actual `claude_client.get_client()` and `EXTRACTION_MODEL` from the
codebase, not a hand-rolled client. That way the model routing you prototype
against is identical to what production uses.

`load_dotenv` must run **before** importing `app.config`: the settings object
snapshots env vars when it is constructed, so the keys have to be in the
environment first. (That exact ordering trap is why a settings-vs-env test bug
slipped into PR #6 — see the build tracker.)


In [ ]:
import sys
from pathlib import Path

# Works whether the kernel starts in backend/notebooks/ or backend/
cwd = Path.cwd()
backend = cwd if (cwd / "app").is_dir() else cwd.parent
assert (backend / "app").is_dir(), f"Run from backend/ or backend/notebooks/. cwd={cwd}"
sys.path.insert(0, str(backend))

from dotenv import load_dotenv
load_dotenv(backend / ".env")          # <-- before any `app.*` import

from app.services import claude_client
from app.services.claude_client import EXTRACTION_MODEL
from app.utils.json_helpers import strip_json_fences   # reuse the real fence stripper

client = claude_client.get_client()
print("model :", EXTRACTION_MODEL)
print("client:", type(client).__name__)

## Step 2 — a sample document

In production this text comes out of OCR (`PyMuPDF` / `pytesseract`). On the
bench we paste a short, **obviously synthetic** record so the loop is fast and
free of file handling. Swap in a real OCR dump when you want to test a hard case.

`FIELDS` is a handful of real entries copied from the parcel/deed schema in
`app/seeds/document_schemas.py` — same shape (`name`, `type`, `description`) the
engine feeds to Claude.


In [ ]:
SAMPLE_TEXT = """
COUNTY AUDITOR - PROPERTY RECORD CARD
Parcel Number: 003-07-26-014-000
County: Centerburg

Location:  Owner of Record: RIVERSIDE HOLDINGS LLC
           Property Address: 142 Maple Ridge Rd, Centerburg

Deeded Owner Address: RIVERSIDE HOLDINGS LLC, 9000 Corporate Way, Suite 300
Tax Payer Address:    SAMPLE MANAGEMENT CO, PO Box 77, Centerburg

Legal: MAPLE RIDGE SUBDIVISION LOT 14    Legal Acres: 1.25
Land Use: 510 Single Family Dwelling
"""

# A few fields lifted from the real schema seed. The engine sends exactly this shape.
FIELDS = [
    {"name": "parcel_number",   "type": "id_number", "description": "Official county parcel identifier at the top of the document"},
    {"name": "owner_name",      "type": "name",      "description": "Current owner of record as shown in the Location section"},
    {"name": "property_address","type": "address",   "description": "Physical property address"},
    {"name": "taxpayer_name",   "type": "name",      "description": "Owner name from the Tax Payer Address section"},
    {"name": "owner_is_trust",  "type": "boolean",   "description": "True if owner name contains TRUSTEE, TRUST, or similar trust language"},
    {"name": "owner_entity_type","type": "text",     "description": "individual, LLC, nonprofit, trust, or government"},
]

## Step 3 — build the prompt

This mirrors the `static_prompt` in `_extract_batch` (extraction_engine.py),
trimmed to one text block. Production splits it into a cached static block + an
uncached document block for prompt caching — irrelevant while prototyping, so we
keep it as one string and stay readable.

`extract()` calls Claude and runs the response through the real
`strip_json_fences` helper, so you hit the same JSON-parsing path the engine does.


In [ ]:
import json

def build_prompt(fields, preamble="Extract the following fields from this property record."):
    fields_desc = "\n".join(f"- {f['name']} ({f['type']}): {f['description']}" for f in fields)
    return f"""{preamble}

Extract ONLY these {len(fields)} fields:
{fields_desc}

Rules:
- Use EXACTLY these JSON key names: "field_name", "field_value", "field_type", "confidence"
- field_value must be a string or null — never a number or boolean
- confidence: your certainty in the extraction (0.0 to 1.0)
- Respond with JSON only — no markdown, no explanation

Required format:
{{"extractions": [
    {{"field_name": "exact_name", "field_value": "text or null", "field_type": "text", "confidence": 0.9}}
]}}"""

def extract(text, fields, **kw):
    prompt = build_prompt(fields, **kw)
    resp = client.messages.create(
        model=EXTRACTION_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": f"{prompt}\n\nDocument text:\n{text}"}],
    )
    return json.loads(strip_json_fences(resp.content[0].text))

print(build_prompt(FIELDS))   # eyeball the actual prompt before sending it

## Step 4 — run it and *look*

The output cell is the whole point. Read the values. Did it catch that the
taxpayer differs from the owner? Did `owner_is_trust` come back false for an LLC
(correct — an LLC is not a trust)? This is the reacting-to-results loop a plain
script can't give you.


In [ ]:
result = extract(SAMPLE_TEXT, FIELDS)
for e in result["extractions"]:
    print(f"{e['field_name']:18} = {str(e['field_value'])[:32]:32}  (conf {e['confidence']})")

## Step 5 — the loop: change one thing, re-run, compare

Suppose the model is shaky on `owner_entity_type`. Tighten the description and
re-run *only this cell*. Compare against Step 4's output. You are tuning by eye —
exactly what the bench is for. Try a few variants until it is solid.


In [ ]:
FIELDS_V2 = [dict(f) for f in FIELDS]
for f in FIELDS_V2:
    if f["name"] == "owner_entity_type":
        f["description"] = (
            "One of exactly: individual, LLC, nonprofit, trust, government. "
            "Infer from the owner name suffix (LLC, INC, TRUST, etc.)."
        )

result_v2 = extract(SAMPLE_TEXT, FIELDS_V2)
for e in result_v2["extractions"]:
    print(f"{e['field_name']:18} = {str(e['field_value'])[:32]:32}  (conf {e['confidence']})")

## Step 6 — bench → product

Once a prompt or field description is solid here, it does **not** stay in the
notebook. It graduates into the real code:

| What you tuned on the bench | Where it lives in production |
|------|------|
| A field's `description` | `app/seeds/document_schemas.py` (the schema seed row) |
| The `preamble` text | `schema.extraction_prompt` — a column on the `document_schemas` row |
| The prompt scaffolding / rules | Already in `_extract_batch` — usually no change needed |

Then **delete this notebook** (or leave it untracked). The repo has zero
committed notebooks on purpose: a notebook records an experiment, not the
product. Git has the product; your eyes had the experiment.
